In [ ]:
from sqlite3 import connect
from glob import glob
import numpy as np
EX = {i:(alias or i,bool(ex&0x4802040),name)for i,alias,ex,name in connect('cards.cdb').execute('select datas.id,alias,type,name from datas join texts on datas.id=texts.id')}
LF = dict(map(int,i.split()[:2])for i in open('lflist.conf').read().split('!')[1].split('\n')if'-'in i)
MAP = {}
DECK = np.array([(j,MAP.setdefault(EX[i][0],len(MAP)))for j,i in enumerate(glob('deck/**/*.ydk',recursive=1,include_hidden=1))for i in[[int(i)for i in open(i).read().split('!')[0].split()if i.isdigit()and int(i)in EX]]if len(i)>=40 for i in i],'H')
NAME = np.array([EX[i][2]for i in MAP],'U8')
EX = np.array([EX[i][1]for i in MAP])
EXS = EX.nonzero()[0].astype('H')
MAINS = (~EX).nonzero()[0].astype('H')
LF = np.array([LF.get(i,3)for i in MAP],'f')
MAP_ = np.array(list(MAP),'I')
DECK = np.add.at(_:=np.zeros((DECK[-1,0]+1,len(EX)),'f'),tuple(DECK.T),1)or _
DECK += 1e-9
DECK /= DECK.sum(1,keepdims=1)
DECKLOG = np.log(DECK)
DECK = np.asfortranarray(DECK)
WEIGHT = DECK.sum(0)

In [ ]:
from scipy.special import softmax
MAIN = 40
NORM = WEIGHT**-.5
deck = np.zeros_like(DECK[:1])
deck[0,[MAP[23626223]]] = 3
deck[0,[]] = -3
cump = np.zeros(1,'f')
ex = 15-(deck.clip(0)@EX).astype('B')
main = 55-int(deck.clip(0).sum())-ex
for _ in range(main[0]+ex[0]):
    q = deck*NORM
    q_ = q.clip(0)
    q = q/q_.sum(1,keepdims=1)
    p = softmax(q@DECKLOG.T)@DECK
    cump = cump[:,None]+p/np.linalg.norm(p,2,1,1)-q_/np.linalg.norm(q_,2,1,1)
    cump[(deck<0)|(deck==LF)] = 0.
    cump[np.logical_not(main).nonzero()[0][:,None],MAINS] = 0.
    cump[np.logical_not(ex).nonzero()[0][:,None],EXS] = 0.
    i,j = np.unravel_index(cump.argpartition(-16,None)[-16:],p.shape)
    deck = deck[i]
    deck[range(len(i)),j] += 1.
    deck,k = np.unique(deck.view(('V',len(EX)*4)),1)
    deck = deck[:,None].view('f')
    i = i[k]
    j = j[k]
    cump = cump[i,j]
    ex = ex[i]
    ex -= EX[j]
    main = main[i]
    main -= ~EX[j]
    i = cump.argmax()
    j, = (deck[i]>0).nonzero()
    print(*(j for j in j for j in(NAME[j],deck[i,j])))
print(*(MAP_[j]for j in j for _ in range(int(deck[i,j]))),sep='\n',file=open('deck/t.ydk','w'))